OK WHAT WE'RE DOING IN THIS NOTEBOOK: ADDING THE REGULARS

I am going to:

1) Combined geojson -> only the non-lines (polygons)
2) polygons df to postings df -> each row associated with a posting id
3) postings gf to  full postings df -> all the info for each posting from the sweeps_df added to the postings df
4) postings df to polydots -> each row is given polydots for where the map will load to
5) polydots to clickable polydots -> each row is given a poly dot to where they'll go when clicked

THEN I"M GONNA ADD THIS TO THE WEBSITE AND SEE HOW IT LOOKS

In [1]:
import json
import re
import ast
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString, shape
import shapely
import math
import geopandas as gpd
import pandas as pd
from shapely.ops import substring, unary_union, linemerge
from shapely.geometry import Point, MultiLineString
import shapely

In [ ]:
#load in the data

with open('../data/combined_verified_sweeps.geojson') as f:
    raw_geojson = json.load(f)

sweep_df = pd.read_csv('../data/final_sweep_df.csv')

print(f"Raw geojson features: {len(raw_geojson['features'])}")
print(f"sweep_df rows: {len(sweep_df)}")

Raw geojson features: 796
sweep_df rows: 2206


In [8]:
# ── STEP 1: Combined GeoJSON → only the polygons ──────────────────────────

with open('../data/combined_verified_sweeps.geojson') as f:
    raw_geojson = json.load(f)

poly_features = [
    f for f in raw_geojson['features']
    if f['geometry']['type'] == 'MultiPolygon'
]

polys_geojson = {
    "type": "FeatureCollection",
    "features": poly_features
}

polys_gdf = gpd.GeoDataFrame.from_features(poly_features, crs="EPSG:4326")

print(f"Total features: {len(raw_geojson['features'])}")
print(f"Polygon features: {len(polys_gdf)}")
print(polys_gdf[['location', 'num_postings', 'sweep_event_ids']].head())

Total features: 796
Polygon features: 169
                                            location  num_postings  \
0                      100 Oak St/Oak St & Fallon St             1   
1  1000 Calcot Place (Homeless camp is on 23rd Av...             4   
2                              10451 MacArthur Blvd.             4   
3              10th & Pine Street / 1022 Pine Street             1   
4                                  10th St & East Dr             1   

                 sweep_event_ids  
0                         [30_0]  
1   [27_33, 27_15, 27_13, 27_11]  
2  [7_4, mergewith7_1, 7_1, 7_0]  
3                         [39_0]  
4                        [31_19]  


In [9]:
# ── STEP 2: Polygons GDF → one row per posting ────────────────────────────

def parse_array_field(val):
    """Handle both proper JSON arrays and Python string representations."""
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        val = re.sub(r'\s+', ' ', val.strip())
        try:
            return json.loads(val)
        except:
            pass
        try:
            return ast.literal_eval(val)
        except:
            pass
        val = val.strip("[]").strip()
        return [v.strip().strip("'\"") for v in val.split() if v.strip()]
    return [val]

rows = []
for _, feat in polys_gdf.iterrows():
    posting_ids = parse_array_field(feat['posting_ids'])
    op_dates    = parse_array_field(feat['operation_dates'])
    sweep_ids   = parse_array_field(feat['sweep_event_ids'])

    for i, pid in enumerate(posting_ids):
        rows.append({
            'location':           feat['location'],
            'posting_id':         int(pid),
            'dot_index':          i,
            'operation_date':     op_dates[i] if i < len(op_dates) else None,
            'sweep_event_id':     sweep_ids[i] if i < len(sweep_ids) else sweep_ids[0],
            'district':           feat['district'],
            'sensitivity_zone':   feat['sensitivity_zone'],
            'intervention_types': feat['intervention_types'],
            'num_postings':       feat['num_postings'],
            'geometry':           feat['geometry'],
        })

postings_from_polys = pd.DataFrame(rows)
print(f"Exploded rows: {len(postings_from_polys)}")
print(postings_from_polys[['location', 'posting_id', 'dot_index', 'operation_date']].head(8))

Exploded rows: 476
                                            location  posting_id  dot_index  \
0                      100 Oak St/Oak St & Fallon St        1135          0   
1  1000 Calcot Place (Homeless camp is on 23rd Av...        1173          0   
2  1000 Calcot Place (Homeless camp is on 23rd Av...        1667          1   
3  1000 Calcot Place (Homeless camp is on 23rd Av...        1715          2   
4  1000 Calcot Place (Homeless camp is on 23rd Av...        1728          3   
5                              10451 MacArthur Blvd.         113          0   
6                              10451 MacArthur Blvd.        1903          1   
7                              10451 MacArthur Blvd.        1909          2   

                          operation_date  
0    November 18 2024 - November 18 2024  
1      October 17 2024 - October 17 2024  
2    November 14 2022 - November 14 2022  
3      October 10 2022 - October 10 2022  
4  September 28 2022 - September 28 2022  
5      Octo

In [13]:
# should be 479
print(len(postings_from_polys))

# each location's row count should match its num_postings value
check = postings_from_polys.groupby('location').agg(
    rows=('posting_id', 'count'),
    num_postings=('num_postings', 'first')
)
check['match'] = check['rows'] == check['num_postings']
print(check['match'].value_counts())
print(check[~check['match']])  # should be empty

dupes = postings_from_polys['posting_id'].duplicated()
print(f"Duplicate posting_ids: {dupes.sum()}")  # should be 0

print(postings_from_polys['posting_id'].isna().sum())  # should be 0

476
match
True    169
Name: count, dtype: int64
Empty DataFrame
Columns: [rows, num_postings, match]
Index: []
Duplicate posting_ids: 0
0


In [14]:
# ── STEP 3: Add sweep metadata from sweep_df ──────────────────────────────

sweep_cols = [
    'posting_id', 'unique_location_id',
    'operation_start_date', 'operation_end_date',
    'intervention', 'before_after_grants_pass',
    'mid_lat', 'mid_lng'
]

sweep_subset = sweep_df[sweep_cols].drop_duplicates('posting_id')

postings_df = postings_from_polys.merge(
    sweep_subset,
    on='posting_id',
    how='left'
)

postings_df = postings_df.drop_duplicates(subset='posting_id', keep='first')

matched   = postings_df['unique_location_id'].notna().sum()
unmatched = postings_df['unique_location_id'].isna().sum()
print(f"Matched: {matched} / Unmatched: {unmatched}")

if unmatched > 0:
    print("Unmatched locations:")
    print(postings_df[postings_df['unique_location_id'].isna()]['location'].unique())

Matched: 476 / Unmatched: 0


wait i went and did the dot placement outside of this notebook - I'm gonna reupload the geojson here. 

In [15]:
# Step 4: Load polygon dots (generated externally)
polygon_dots_gdf = gpd.read_file('../data/polygon_dots_regulars.geojson')
dots_gdf = gpd.read_file('../data/posting_dots_4_2.geojson')
# Step 5: No click offset needed — clicked = load for polygons (already set)

# Step 6: Combine with line-based dots
combined_dots = pd.concat([dots_gdf, polygon_dots_gdf], ignore_index=True)
combined_gdf = gpd.GeoDataFrame(combined_dots, geometry='geometry', crs='EPSG:4326')
print(f"Total dots: {len(combined_gdf)}")

# Step 7: Export
combined_gdf.to_file('../data/all_posting_dots.geojson', driver='GeoJSON')

Skipping field intervention_types: unsupported OGR type: 5
Skipping field district: unsupported OGR type: 5
Skipping field sensitivity_zone: unsupported OGR type: 5


Total dots: 2200
